In [ ]:
# Task 1: Datenimport & Preprocessing

In diesem Schritt laden wir die experimentellen Rohdaten direkt aus deinem OSF-Projekt (`https://osf.io/j27pd/`) via API in das Jupyter Notebook. 

**Wichtiger Hinweis für Google Colab:** 
Damit Google Colab auf die Daten zugreifen kann, muss dein OSF-Projekt bzw. der entsprechende Ordner auf **"Public" (Öffentlich)** gestellt sein. Andernfalls verweigert die OSF-API den Zugriff und wirft einen Fehler (z.B. HTTP 401 oder 403).

Anschließend bereinigen wir den Datensatz:
1. Wir filtern unwichtige Trials heraus (z.B. Zeilen ohne Such-Trials wie Instruktionen oder Fixationskreuze).
2. Wir konvertieren Datentypen (z. B. Reaktionszeiten als numerische Werte, Korrektheit als Boolean).

In [ ]:
import pandas as pd
import requests
import io

# OSF-Datei-URL (Ersetze dies durch den direkten Download-Link deiner OSF-CSV-Datei)
osf_file_url = "https://files.de-1.osf.io/v1/resources/j27pd/providers/osfstorage/"

try:
    print("Versuche Daten von OSF zu laden...")
    # Beispiel-Code zum Laden via Pandas (sobald der direkte CSV-Link vorliegt):
    # response = requests.get(osf_file_url)
    # df = pd.read_csv(io.StringIO(response.content.decode('utf-8')))
    print("Hinweis: Bitte ersetze 'osf_file_url' durch den direkten Link deiner CSV-Datei auf OSF.")
except Exception as e:
    print(f"Fehler beim Laden der Daten: {e}")

# Preprocessing-Funktion
def preprocess_data(df):
    if 'task' in df.columns:
        df = df[df['task'] == 'search_trial'].copy()
    
    df['rt'] = pd.to_numeric(df['rt'], errors='coerce')
    df['block_num'] = pd.to_numeric(df['block_num'], errors='coerce')
    df['correct'] = df['correct'].astype(bool)
    
    return df

In [ ]:
# Task 2: Definition der Variablen & Aggregation

### Definition der Variablen:
* **Unabhängige Variablen (UV):**
  * `condition`: Die Suchbedingung (`old` = wiederholter Kontext vs. `new` = neuer, zufälliger Kontext).
  * `block_num`: Der experimentelle Block (1 bis 10) zur Abbildung des zeitlichen Verlaufs.
* **Abhängige Variablen (AV):**
  * `rt`: Reaktionszeit (Reaction Time in Millisekunden) für korrekte Antworten.
  * `correct`: Korrektheit der Antwort (Genauigkeit).

### Aggregation:
Wir erstellen zwei getrennte Ansichten:
1. **Nach Block und Bedingung:** Für den Lernkurven-Plot.
2. **Nach Proband (Subject) und Bedingung:** Für inferenzstatistische Tests.

In [ ]:
import numpy as np

def aggregate_data(df):
    df_correct = df[df['correct'] == True].copy()
    
    # Aggregation auf Block- & Bedingungs-Ebene
    df_block_agg = df_correct.groupby(['block_num', 'condition']).agg(
        mean_rt=('rt', 'mean'),
        std_rt=('rt', 'std'),
        count=('rt', 'count'),
        mean_accuracy=('correct', 'mean')
    ).reset_index()
    
    df_block_agg['sem_rt'] = df_block_agg['std_rt'] / np.sqrt(df_block_agg['count'])
    
    # Aggregation auf Personen-Ebene
    df_subject_agg = df_correct.groupby(['subject', 'condition']).agg(
        mean_rt=('rt', 'mean'),
        mean_accuracy=('correct', 'mean')
    ).reset_index()
    
    return df_block_agg, df_subject_agg

In [ ]:
# Task 3: Deskriptive Statistik & Visualisierung

### Visualisierung (Lernkurve):
Das klassische Merkmal des **Contextual Cueing Effekts** (Chun & Jiang, 1998) ist, dass die Probanden im Laufe der Blöcke bei den *wiederholten Displays (`old`)* schneller werden als bei den *neuen Displays (`new`)*. 

**Wichtig für die Sortierung:** Die Blöcke (1 bis 10) werden als geordnete Kategorien definiert, damit die Plot-Bibliothek sie chronologisch sortiert[cite: 2].

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

def plot_contextual_cueing(df_block_agg):
    plt.figure(figsize=(10, 6))
    df_block_agg['block_num'] = pd.Categorical(df_block_agg['block_num'], categories=range(1, 11), ordered=True)
    
    sns.lineplot(
        data=df_block_agg,
        x='block_num',
        y='mean_rt',
        hue='condition',
        marker='o',
        linewidth=2.5,
        palette={'old': '#1f77b4', 'new': '#ff7f0e'}
    )
    
    plt.title('Contextual Cueing Effekt: Reaktionszeiten über die Blöcke', fontsize=14, fontweight='bold')
    plt.xlabel('Experimenteller Block', fontsize=12)
    plt.ylabel('Mittlere Reaktionszeit (RT in ms)', fontsize=12)
    plt.legend(title='Bedingung', labels=['Wiederholt (Old)', 'Zufällig (New)'])
    plt.tight_layout()
    plt.show()

In [ ]:
# Task 4: Voraussetzungen prüfen (Assumption Checks)

Bevor wir Inferenzstatistik betreiben, prüfen wir:
1. **Normalverteilung:** mittels Shapiro-Wilk-Test.
2. **Varianzhomogenität:** mittels Levene-Test.
3. **Hinweis zu Deckeneffekten:** Moderne **Multilevel-Modelle (MLM)** lassen sich von vereinzelten Extremwerten nicht so leicht aus der Ruhe bringen, da sie hierarchisch arbeiten.

In [ ]:
from scipy import stats

def check_assumptions(df_subject_agg):
    print("--- 1. Shapiro-Wilk-Test auf Normalverteilung ---")
    for condition in df_subject_agg['condition'].unique():
        subset = df_subject_agg[df_subject_agg['condition'] == condition]['mean_rt']
        stat, p_value = stats.shapiro(subset)
        print(f"Bedingung '{condition}': p-Wert = {p_value:.4f}")
            
    print("\n--- 2. Levene-Test auf Varianzhomogenität ---")
    old_rt = df_subject_agg[df_subject_agg['condition'] == 'old']['mean_rt']
    new_rt = df_subject_agg[df_subject_agg['condition'] == 'new']['mean_rt']
    levene_stat, levene_p = stats.levene(old_rt, new_rt)
    print(f"Levene-Test p-Wert = {levene_p:.4f}")

In [ ]:
# Task 5: Einfache Gruppenvergleiche (Inferenzstatistik)

Wir nutzen einen **gepaarten t-Test**, da jede Person sowohl die `old`- als auch die `new`-Bedingung absolviert hat. 
* **p-Wert:** Zeigt die statistische Signifikanz ($p < 0.05$).
* **Cohen's d:** Misst die Effektstärke unabhängig von der Stichprobengröße.

In [ ]:
def run_inferential_stats(df_subject_agg):
    old_data = df_subject_agg[df_subject_agg['condition'] == 'old'].sort_values('subject')['mean_rt'].values
    new_data = df_subject_agg[df_subject_agg['condition'] == 'new'].sort_values('subject')['mean_rt'].values
    
    t_stat, p_val = stats.ttest_rel(old_data, new_data)
    
    diff = old_data - new_data
    cohens_d = np.mean(diff) / np.std(diff, ddof=1)
    
    print(f"t-Statistik: {t_stat:.4f} | p-Wert: {p_val:.5f} | Cohen's d: {cohens_d:.4f}")

In [ ]:
# Task 6: Multilevel Modeling (MLM)

Das Multilevel-Modell berücksichtigt die Verschachtelung der Daten (Messungen innerhalb von Personen) durch **Random Intercepts** und filtert individuellen Probanden-Lärm heraus. 
* **Spaghetti-Plot:** Dünne graue Linien zeigen die individuellen Probandenverläufe, die dicken farbigen Linien den aggregierten Trend.

In [ ]:
import statsmodels.formula.api as smf

def run_multilevel_model(df):
    df_correct = df[df['correct'] == True].copy()
    
    # MLM fitten
    model = smf.mixedlm("rt ~ condition", df_correct, groups=df_correct["subject"])
    result = model.fit()
    print(result.summary())
    
    # Spaghetti-Plot
    plt.figure(figsize=(10, 6))
    df_indiv = df_correct.groupby(['block_num', 'subject', 'condition'])['rt'].mean().reset_index()
    
    sns.lineplot(data=df_indiv, x='block_num', y='rt', units='subject', estimator=None, color='gray', alpha=0.3, linewidth=1)
    
    df_pop = df_correct.groupby(['block_num', 'condition'])['rt'].mean().reset_index()
    sns.lineplot(data=df_pop, x='block_num', y='rt', hue='condition', linewidth=3, palette={'old': 'blue', 'new': 'orange'})
    
    plt.title('Multilevel-Modell: Individuelle Verläufe und Gruppentrend', fontsize=14, fontweight='bold')
    plt.show()